# cancer-recon-apoptosis — RUNG 4 / Step-5: logic-gate recognition designer

Solves the wall Step-3 + RUNG-3 keep hitting: **no single tumour-exclusive antigen exists**. Searches **AND / AND-NOT** antigen combinations and scores each by single-cell selectivity — *is any normal cell, especially in heart/brain/kidney, double-positive?*

Two stages: **(1) validate the engine** on biological ground truth (CPU, seconds, no data) → must score HER2-alone unsafe on heart, Tmod gates selective, bulk-trap exposed; **(2) real discovery** on CELLxGENE single-cell atlases (the antigen pool that FAILED Step-3, now gated).

**Ceiling:** transcript-level HYPOTHESIS. mRNA≠surface protein (CITE-seq confirms); co-localisation≠a working circuit (wet-lab); recognition is a separate axis, never multiplied with RUNG-1/2/3. 'No clean gate exists' is a valid result.

In [ ]:
#@title Cell 1 — clone / pull repo
import os
from pathlib import Path
REPO = Path('/content/cancer-recon-apoptosis')
if REPO.exists():
    !cd {REPO} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recon-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — start run log
import sys, importlib.util
from scripts.runlog import new_log, run_logged, finalize
RUNLOG = new_log('rung4_logicgate', repo_dir='.')
print('[CELL 2] ✓')

In [ ]:
#@title Cell 3 — VALIDATE the engine on biological ground truth (CPU, no data needed)
import sys
from scripts.runlog import new_log, run_logged
RUNLOG = globals().get('RUNLOG') or new_log('rung4_logicgate', repo_dir='.')
rc = run_logged([sys.executable, '-u', 'scripts/20_logicgate_calibration.py'], RUNLOG)
print(f'[CELL 3] exit={rc}', '✓ RUN-TRUST passed' if rc == 0 else '✗ engine failed validation')
from IPython.display import Image, display
import os
if os.path.exists('runs/rung4_logicgate/rung4_calibration.png'):
    display(Image('runs/rung4_logicgate/rung4_calibration.png'))

In [ ]:
#@title Cell 4 — install CELLxGENE Census (only needed for the real discovery)
import sys, importlib.util
from scripts.runlog import run_logged
for pkg, pip_name in [('cellxgene_census', 'cellxgene-census'), ('scanpy', 'scanpy')]:
    if importlib.util.find_spec(pkg) is None:
        run_logged([sys.executable, '-m', 'pip', 'install', '-q', pip_name], RUNLOG, label=f'pip install {pip_name}')
print('[CELL 4] ✓')

In [ ]:
#@title Cell 5 — REAL discovery: score AND gates on CELLxGENE single-cell atlases
import sys
rc = run_logged([sys.executable, '-u', 'scripts/17_logicgate_data.py'], RUNLOG)
print(f'[CELL 5] exit={rc}')
import json, os
if os.path.exists('runs/rung4_logicgate/rung4_results.json'):
    r = json.load(open('runs/rung4_logicgate/rung4_results.json'))
    print('vital coverage:', r.get('vital_coverage'))
    print('UNAUDITED vital types:', r.get('unaudited_vital_types'))
    print('gates scored:', r.get('n_gates'), '| predicted SELECTIVE:', r.get('n_selective'))
    print('NO clean gate?', r.get('no_clean_gate'))

In [ ]:
#@title Cell 6 — finalize + download run log
from scripts.runlog import new_log, finalize
RUNLOG = globals().get('RUNLOG') or new_log('rung4_logicgate', repo_dir='.')
finalize(RUNLOG)

## What this rung established

- **The selectivity engine is validated** — it re-derives Step-3 (HER2 unsafe on heart), scores known Tmod gates selective, and exposes the bulk-trap (pseudobulk falsely sees ~49% liver co-expression vs ~1% single-cell).
- **Real discovery is transcript-only** — every gate tagged `NO_SINGLECELL_PROTEIN_DATA` until CITE-seq; under-sampled vital types flagged `UNAUDITED` (use snRNA-seq for heart/brain).
- **Selectivity and escape-durability are separate axes** — AND-gating buys safety, pays durability; for a contact death-wave an antigen-negative escaper is unreachable.
- **'No clean gate exists' is a first-class result** — no forced winner.

Next: `scripts/19` (CITE-seq protein cross-check), `scripts/21` (full clonal antigen-loss durability sim), and the wet-lab handoff (CITE-seq co-positivity + flow on primary vital cells + logic-gated co-culture).